# Baselines supervisées (Fashion-MNIST)

Même logique que le notebook 03 original : pour situer les approches à base d'auto-encodeur
(notebooks 01 et 02), il faut un point de comparaison — que donnent des modèles supervisés
classiques, entraînés directement sur les pixels bruts et toutes les étiquettes ?

Deux modèles standards — régression logistique et random forest — réglés par grid search en
validation croisée. Sur T-shirt/top vs Shirt, ces modèles n'ont accès à aucune structure
d'image (juste 784 pixels comme 784 variables indépendantes), donc c'est un bon test de ce
qu'une approche purement tabulaire peut extraire de la paire la plus confondue du dataset.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
from sklearn.metrics import classification_report, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 123

## Données

Même dataset et même découpage que le notebook 02 (80 % train / 20 % test, stratifié) pour que
la comparaison soit juste. Le réglage des hyperparamètres se fait par validation croisée à
l'intérieur du train, donc le test reste totalement à part.

In [2]:
data = fetch_openml("Fashion-MNIST", version=1, as_frame=False, parser="auto")
X_all, y_all = data.data, data.target.astype(int)

mask = np.isin(y_all, [0, 6])
X = X_all[mask]
y = (y_all[mask] == 6).astype(int)  # 0 = T-shirt/top, 1 = Shirt
target_names = np.array(["T-shirt/top", "Shirt"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train :", X_train.shape, "| Test :", X_test.shape)

Train : (11200, 784) | Test : (2800, 784)


## Réglage par grid search

Chaque modèle est encapsulé dans un pipeline. La régression logistique a besoin d'un
`StandardScaler` (elle est sensible à l'échelle des pixels, et sans normalisation `liblinear`
met un temps déraisonnable à converger sur 784 variables) ; la random forest n'en a pas besoin
puisqu'elle travaille par seuils. Le scoring est le ROC-AUC, en validation croisée stratifiée
5 folds.

In [3]:
models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(solver="liblinear", class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "RandomForest": Pipeline([
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
}

param_grids = {
    "LogisticRegression": {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__penalty": ["l1", "l2"],
    },
    "RandomForest": {
        "clf__n_estimators": [50, 100, 200],
        "clf__max_depth": [5, 10, None],
        "clf__min_samples_leaf": [1, 2, 4],
    },
}

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

best_models = {}
grid_results = []
for model_name, pipeline in models.items():
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
    )
    grid_search.fit(X_train, y_train)
    best_models[model_name] = grid_search.best_estimator_
    grid_results.append({
        "Modele": model_name,
        "Best ROC-AUC (CV)": round(grid_search.best_score_, 4),
        "Meilleurs hyperparametres": grid_search.best_params_,
    })

df_grid = pd.DataFrame(grid_results).set_index("Modele")
display(df_grid)

,Best ROC-AUC (CV),Meilleurs hyperparametres
Modele,,
LogisticRegression,0.9284,"{'clf__C': 0.1, 'clf__penalty': 'l1'}"
RandomForest,0.9450,"{'clf__max_depth': None, 'clf__min_samples_lea..."


## Résultats sur le test

In [5]:
rows = []
for model_name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    f1m = f1_score(y_test, y_pred, average="macro")

    print(f"{'='*20} {model_name} {'='*20}")
    print(f"ROC-AUC : {auc:.4f} | F1 (macro) : {f1m:.4f}\n")
    print(classification_report(y_test, y_pred, target_names=target_names))
    rows.append({"Modele": model_name, "ROC-AUC": round(auc, 4), "F1 (macro)": round(f1m, 4)})

pd.DataFrame(rows).set_index("Modele")

==================== LogisticRegression ====================
ROC-AUC : 0.9283 | F1 (macro) : 0.8539

              precision    recall  f1-score   support

 T-shirt/top       0.84      0.87      0.86      1400
       Shirt       0.87      0.83      0.85      1400

    accuracy                           0.85      2800
   macro avg       0.85      0.85      0.85      2800
weighted avg       0.85      0.85      0.85      2800



==================== RandomForest ====================
ROC-AUC : 0.9450 | F1 (macro) : 0.8619

              precision    recall  f1-score   support

 T-shirt/top       0.84      0.90      0.87      1400
       Shirt       0.89      0.82      0.86      1400

    accuracy                           0.86      2800
   macro avg       0.86      0.86      0.86      2800
weighted avg       0.86      0.86      0.86      2800



,ROC-AUC,F1 (macro)
Modele,,
LogisticRegression,0.9283,0.8539
RandomForest,0.9450,0.8619


## Ce que je retiens

Sur pixels bruts, les baselines sont nettement moins solides que sur le cancer : régression
logistique à 0,928 de ROC-AUC (vs 0,993 sur le cancer), random forest à 0,945 (vs 0,997). C'est
attendu — sans aucune notion de structure d'image, chaque pixel est traité comme une variable
indépendante, ce qui est un mauvais point de départ pour distinguer deux vêtements qui ne
diffèrent que par des détails de forme localisés.

C'est justement ce qui rend la comparaison avec le notebook 02 intéressante, contrairement au
cancer où le SSL ne faisait que s'approcher des baselines sans les dépasser : ici, le full
fine-tuning (ROC-AUC ~0,94 quelle que soit la dimension latente) dépasse nettement la régression
logistique (0,928) et rejoint quasiment la random forest (0,945). Le linear probing, lui, reste
en dessous à faible dimension latente (0,867 en latent 2) mais rattrape la régression logistique
dès la dimension 16 (0,931). Sur un dataset assez dur pour qu'un modèle linéaire sur pixels bruts
ne suffise plus, le pré-entraînement non supervisé apporte donc un vrai avantage — exactement
l'hypothèse formulée à la fin du projet original.